# Thu thập Dữ liệu Dân số Đối chiếu từ World Population Review (2024 - 2026)

Notebook này thực hiện thu thập dữ liệu dân số các quốc gia cho giai đoạn 2024 - 2026 từ World Population Review.
Dữ liệu này được lưu trữ trong thư mục `data/raw/` phục vụ mục đích kiểm chứng chéo (validation) với các dự báo của LHQ (UN WPP).

In [1]:
import csv
import html
import json
import re
import time
from datetime import datetime, timezone
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

# Thiết lập đường dẫn và cấu hình
ROOT = Path("..").resolve() if Path(".").resolve().name == "notebooks" else Path(".").resolve()
RAW_DIR = ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

BASE_URL = "https://worldpopulationreview.com"
COUNTRIES_URL = f"{BASE_URL}/countries"
TARGET_YEARS = (2024, 2025, 2026)
USER_AGENT = "population-trends-project/1.0"

COUNTRY_LINK_PATTERN = re.compile(r'href=["\'](?:https://worldpopulationreview\.com)?/countries/([^/"?#]+)["\']')
POPULATION_PATTERN = re.compile(r'\{"year":(2024|2025|2026),"population":([0-9]+(?:\.[0-9]+)?)\}')

OUTPUT_CSV = RAW_DIR / "world-population-review-2024-2026.csv"
OUTPUT_METADATA = RAW_DIR / "world-population-review-metadata.json"

print("Đường dẫn lưu CSV:", OUTPUT_CSV)
print("Đường dẫn lưu Metadata:", OUTPUT_METADATA)


Đường dẫn lưu CSV: /Users/phitaan/Documents/WORKSPACE/TTDLTQ/PROJECT CUỐI KỲ - BASIC/data/raw/world-population-review-2024-2026.csv
Đường dẫn lưu Metadata: /Users/phitaan/Documents/WORKSPACE/TTDLTQ/PROJECT CUỐI KỲ - BASIC/data/raw/world-population-review-metadata.json


## 1. Tải Danh sách Quốc gia từ World Population Review

In [2]:
# Tải trang danh mục quốc gia
print(f"Đang tải trang danh mục: {COUNTRIES_URL}...")
request = Request(COUNTRIES_URL, headers={"User-Agent": USER_AGENT})
with urlopen(request, timeout=20) as resp:
    countries_page = resp.read().decode("utf-8")

# Trích xuất các slug quốc gia
all_slugs = sorted(set(COUNTRY_LINK_PATTERN.findall(countries_page)) - {"by-gdp"})
print(f"Tìm thấy {len(all_slugs)} slug quốc gia/lãnh thổ.")
print("Ví dụ 10 slug đầu tiên:", all_slugs[:10])

Đang tải trang danh mục: https://worldpopulationreview.com/countries...


Tìm thấy 235 slug quốc gia/lãnh thổ.
Ví dụ 10 slug đầu tiên: ['afghanistan', 'albania', 'algeria', 'american-samoa', 'andorra', 'angola', 'anguilla', 'antigua-and-barbuda', 'argentina', 'armenia']


## 2. Đọc Dữ liệu Dân số đã Lưu hoặc Tiến hành Thu thập Mới

In [3]:
# Kiểm tra nếu file đã có sẵn trong data/raw/ thì hiển thị thông tin
if OUTPUT_CSV.exists():
    with OUTPUT_CSV.open(encoding="utf-8") as f:
        existing_rows = list(csv.DictReader(f))
    print(f"File raw đã tồn tại với {len(existing_rows):,} bản ghi.")
    print("Mẫu 3 bản ghi đầu tiên:")
    for r in existing_rows[:3]:
        print(" ", r)
else:
    print("Bắt đầu thu thập dữ liệu mới từ web...")
    records = []
    errors = []
    for idx, slug in enumerate(all_slugs, 1):
        country_name = " ".join(word.capitalize() for word in slug.replace("-", " ").split())
        url = f"{BASE_URL}/countries/{slug}"
        if idx % 50 == 0 or idx == len(all_slugs):
            print(f"Tiến độ: {idx}/{len(all_slugs)} quốc gia")
        try:
            req = Request(url, headers={"User-Agent": USER_AGENT})
            with urlopen(req, timeout=15) as r:
                html_text = html.unescape(r.read().decode("utf-8"))
            year_vals = {}
            for y_str, pop_str in POPULATION_PATTERN.findall(html_text):
                year_vals.setdefault(int(y_str), round(float(pop_str)))
            for yr in TARGET_YEARS:
                if yr in year_vals:
                    records.append({
                        "entity": country_name,
                        "slug": slug,
                        "code": slug,
                        "year": yr,
                        "population": year_vals[yr],
                        "source_url": url,
                    })
            time.sleep(0.05)
        except Exception as e:
            errors.append({"slug": slug, "url": url, "error": str(e)})

    if records:
        fields = ["entity", "slug", "code", "year", "population", "source_url"]
        with OUTPUT_CSV.open("w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=fields)
            writer.writeheader()
            writer.writerows(records)
        print(f"Đã lưu thành công: {OUTPUT_CSV}")

File raw đã tồn tại với 705 bản ghi.
Mẫu 3 bản ghi đầu tiên:
  {'entity': 'Afghanistan', 'slug': 'afghanistan', 'code': 'afghanistan', 'year': '2024', 'population': '42889085', 'source_url': 'https://worldpopulationreview.com/countries/afghanistan'}
  {'entity': 'Afghanistan', 'slug': 'afghanistan', 'code': 'afghanistan', 'year': '2025', 'population': '43844111', 'source_url': 'https://worldpopulationreview.com/countries/afghanistan'}
  {'entity': 'Afghanistan', 'slug': 'afghanistan', 'code': 'afghanistan', 'year': '2026', 'population': '45047069', 'source_url': 'https://worldpopulationreview.com/countries/afghanistan'}
